In [0]:
import requests
from time import sleep
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import dlt

---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
File <command-6356111632306832>, line 5
      3 from time import sleep
      4 from pyspark.sql import functions as F
----> 5 import dlt

ModuleNotFoundError: No module named 'dlt'

In [0]:
lst = ["IBM", "META"]
daily_info = []
stock_info = []
company_info = []

# replace the "demo" apikey below with your own key from https://www.alphavantage.co/support/#api-key
for i in range(len(lst)):
    daily_url = "https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol=" + lst[i] + "&interval=5min&apikey=YOUR_API_KEY"
    stock_url = 'https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol=' + lst[i] + '&apikey=YOUR_API_KEY'
    company_url = 'https://www.alphavantage.co/query?function=OVERVIEW&symbol=' + lst[i] + '&apikey=YOUR_API_KEY'

    daily_r = requests.get(daily_url)
    sleep(5)
    stock_r = requests.get(stock_url)
    sleep(5)
    company_r = requests.get(company_url)

    daily_info.append(daily_r.json())
    stock_info.append(stock_r.json())
    company_info.append(company_r.json())
    sleep(10)

In [0]:
@dlt.table(
    name="bronze_daily_prices",
    comment="Raw daily OHLCV data from Alpha Vantage TIME_SERIES_DAILY endltoint"
)
def bronze_daily_prices():
    rows = []
    for i, symbol in enumerate(lst):
        for date_str, vals in daily_info[i].get("Time Series (Daily)", {}).items():
            rows.append({
                "symbol":     symbol,
                "trade_date": date_str,
                "raw_open":   vals.get("1. open"),
                "raw_high":   vals.get("2. high"),
                "raw_low":    vals.get("3. low"),
                "raw_close":  vals.get("4. close"),
                "raw_volume": vals.get("5. volume"),
            })
    return spark.createDataFrame(rows).withColumn("ingestion_timestamp", F.current_timestamp())


@dlt.table(
    name="bronze_quotes",
    comment="Raw latest ticker quote from Alpha Vantage GLOBAL_QUOTE endltoint"
)
def bronze_quotes():
    rows = []
    for i, symbol in enumerate(lst):
        q = stock_info[i].get("Global Quote", {})
        rows.append({
            "symbol":             symbol,
            "raw_open":           q.get("02. open"),
            "raw_high":           q.get("03. high"),
            "raw_low":            q.get("04. low"),
            "raw_price":          q.get("05. price"),
            "raw_volume":         q.get("06. volume"),
            "latest_trading_day": q.get("07. latest trading day"),
            "raw_prev_close":     q.get("08. previous close"),
            "raw_change":         q.get("09. change"),
            "raw_change_pct":     q.get("10. change percent"),
        })
    return spark.createDataFrame(rows).withColumn("ingestion_timestamp", F.current_timestamp())


@dlt.table(
    name="bronze_company_info",
    comment="Raw company fundamentals from Alpha Vantage OVERVIEW endltoint"
)
def bronze_company_info():
    rows = []
    for i, symbol in enumerate(lst):
        ov = company_info[i]
        rows.append({
            "symbol":             symbol,
            "name":               ov.get("Name"),
            "asset_type":         ov.get("AssetType"),
            "exchange":           ov.get("Exchange"),
            "currency":           ov.get("Currency"),
            "country":            ov.get("Country"),
            "sector":             ov.get("Sector"),
            "industry":           ov.get("Industry"),
            "description":        ov.get("Description"),
            "fiscal_year_end":    ov.get("FiscalYearEnd"),
            "latest_quarter":     ov.get("LatestQuarter"),
            "raw_market_cap":     ov.get("MarketCapitalization"),
            "raw_pe_ratio":       ov.get("PERatio"),
            "raw_peg_ratio":      ov.get("PEGRatio"),
            "raw_eps":            ov.get("EPS"),
            "raw_dividend_yield": ov.get("DividendYield"),
            "raw_52wk_high":      ov.get("52WeekHigh"),
            "raw_52wk_low":       ov.get("52WeekLow"),
        })
    return spark.createDataFrame(rows).withColumn("ingestion_timestamp", F.current_timestamp())

---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
File <command-6405983144451424>, line 1
----> 1 import dlt# Bronze Layer — Raw data ingested from Alpha Vantage API
      4 @dlt.table(
      5     name="bronze_daily_prices",
      6     comment="Raw daily OHLCV data from Alpha Vantage TIME_SERIES_DAILY endpoint"
      7 )
      8 def bronze_daily_prices():
      9     rows = []

ModuleNotFoundError: No module named 'dlt'

In [0]:
# Silver Layer — Enriched and reformatted tables from bronze

@dlt.table(
    name="silver_daily_prices",
    comment="Typed and cleaned daily OHLCV prices with daily_range derived column"
)
@dlt.expect_or_drop("valid_close",  "close > 0")
@dlt.expect_or_drop("valid_volume", "volume > 0")
def silver_daily_prices():
    return (
        dlt.read("bronze_daily_prices")
        .withColumn("trade_date",  F.to_date("trade_date",  "yyyy-MM-dd"))
        .withColumn("open",        F.col("raw_open").cast("double"))
        .withColumn("high",        F.col("raw_high").cast("double"))
        .withColumn("low",         F.col("raw_low").cast("double"))
        .withColumn("close",       F.col("raw_close").cast("double"))
        .withColumn("volume",      F.col("raw_volume").cast("long"))
        .withColumn("daily_range", F.col("high") - F.col("low"))
        .drop("raw_open", "raw_high", "raw_low", "raw_close", "raw_volume")
    )


@dlt.table(
    name="silver_quotes",
    comment="Typed latest ticker quote with parsed change percentage"
)
@dlt.expect_or_drop("valid_price", "price > 0")
def silver_quotes():
    return (
        dlt.read("bronze_quotes")
        .withColumn("trade_date",  F.to_date("latest_trading_day", "yyyy-MM-dd"))
        .withColumn("open",        F.col("raw_open").cast("double"))
        .withColumn("high",        F.col("raw_high").cast("double"))
        .withColumn("low",         F.col("raw_low").cast("double"))
        .withColumn("price",       F.col("raw_price").cast("double"))
        .withColumn("volume",      F.col("raw_volume").cast("long"))
        .withColumn("prev_close",  F.col("raw_prev_close").cast("double"))
        .withColumn("change",      F.col("raw_change").cast("double"))
        .withColumn("change_pct",  F.regexp_replace("raw_change_pct", "%", "").cast("double"))
        .drop("latest_trading_day", "raw_open", "raw_high", "raw_low", "raw_price",
              "raw_volume", "raw_prev_close", "raw_change", "raw_change_pct")
    )


@dlt.table(
    name="silver_company_info",
    comment="Typed company fundamentals with cast financial metrics"
)
def silver_company_info():
    return (
        dlt.read("bronze_company_info")
        .withColumn("latest_quarter", F.to_date("latest_quarter", "yyyy-MM-dd"))
        .withColumn("market_cap",     F.col("raw_market_cap").cast("long"))
        .withColumn("pe_ratio",       F.col("raw_pe_ratio").cast("double"))
        .withColumn("peg_ratio",      F.col("raw_peg_ratio").cast("double"))
        .withColumn("eps",            F.col("raw_eps").cast("double"))
        .withColumn("dividend_yield", F.col("raw_dividend_yield").cast("double"))
        .withColumn("high_52wk",      F.col("raw_52wk_high").cast("double"))
        .withColumn("low_52wk",       F.col("raw_52wk_low").cast("double"))
        .drop("raw_market_cap", "raw_pe_ratio", "raw_peg_ratio", "raw_eps",
              "raw_dividend_yield", "raw_52wk_high", "raw_52wk_low")
    )

In [0]:
# Gold Layer — Price and Volume Trend Analysis

@dlt.table(
    name="gold_price_trends",
    comment="Daily price change and pct change vs. 7, 30, and 90-day prior close per ticker"
)
def gold_price_trends():
    df = dlt.read("silver_daily_prices")

    w   = Window.partitionBy("symbol").orderBy("trade_date")
    w7  = Window.partitionBy("symbol").orderBy("trade_date").rowsBetween(-6,  0)
    w30 = Window.partitionBy("symbol").orderBy("trade_date").rowsBetween(-29, 0)
    w90 = Window.partitionBy("symbol").orderBy("trade_date").rowsBetween(-89, 0)

    prev  = F.lag("close", 1).over(w)
    f_7d  = F.first("close").over(w7)
    f_30d = F.first("close").over(w30)
    f_90d = F.first("close").over(w90)

    return (
        df.select("symbol", "trade_date", "close")
        .withColumn("prev_close",    prev)
        # Daily change
        .withColumn("daily_chg",     F.col("close") - F.col("prev_close"))
        .withColumn("daily_pct_chg", (F.col("close") - F.col("prev_close")) / F.col("prev_close") * 100)
        # 7-day window
        .withColumn("chg_7d",        F.col("close") - f_7d)
        .withColumn("pct_chg_7d",    (F.col("close") - f_7d) / f_7d * 100)
        # 30-day window
        .withColumn("chg_30d",       F.col("close") - f_30d)
        .withColumn("pct_chg_30d",   (F.col("close") - f_30d) / f_30d * 100)
        # 90-day window
        .withColumn("chg_90d",       F.col("close") - f_90d)
        .withColumn("pct_chg_90d",   (F.col("close") - f_90d) / f_90d * 100)
        .drop("prev_close")
    )


@dlt.table(
    name="gold_volume_trends",
    comment="Daily volume with rolling avg and totals over 7, 30, and 90-day windows per ticker"
)
def gold_volume_trends():
    df = dlt.read("silver_daily_prices")

    w7  = Window.partitionBy("symbol").orderBy("trade_date").rowsBetween(-6,  0)
    w30 = Window.partitionBy("symbol").orderBy("trade_date").rowsBetween(-29, 0)
    w90 = Window.partitionBy("symbol").orderBy("trade_date").rowsBetween(-89, 0)

    return (
        df.select("symbol", "trade_date", "volume")
        .withColumn("avg_volume_7d",     F.avg("volume").over(w7))
        .withColumn("avg_volume_30d",    F.avg("volume").over(w30))
        .withColumn("avg_volume_90d",    F.avg("volume").over(w90))
        .withColumn("total_volume_7d",   F.sum("volume").over(w7))
        .withColumn("total_volume_30d",  F.sum("volume").over(w30))
        .withColumn("total_volume_90d",  F.sum("volume").over(w90))
        .withColumn("vol_vs_avg_7d_pct",
            (F.col("volume") - F.col("avg_volume_7d")) / F.col("avg_volume_7d") * 100
        )
    )